# Milestone 1 – Military Data Collection and Preparation

This notebook implements the first stage of the **Unified Military Analytics and Comparison Dashboard** project.

The goal of this milestone is to collect military statistics for multiple countries using automated web scraping, parse the extracted HTML data, clean the values, and store them in a structured dataset for further analysis.

Main tasks implemented in this notebook:

- Scraping military statistics webpages
- Storing raw HTML pages
- Parsing metric data using BeautifulSoup
- Cleaning numeric values and units
- Creating a structured dataset
- Exporting the final dataset as CSV

## Step 1: Import Required Libraries

This section imports all required Python libraries used for:

- Sending HTTP requests to websites
- Parsing HTML content
- Data manipulation
- Progress tracking during scraping
- File and directory operations

In [1]:
# Importing required libraries
import requests
import os
from tqdm import tqdm
import time

## Step 2: Creating Directory Structure

Before downloading webpages, directories are created to store the scraped HTML files.

The structure includes:

- `gfp_pages/` → stores the main pages
- `gfp_pages/metrics/` → stores individual metric pages

Saving raw HTML allows debugging and prevents repeated scraping from the website.

In [2]:
# mkdir
os.makedirs("gfp_pages/metrics", exist_ok=True)

## Step 3: Defining Source URLs

In this step, the URLs containing military statistics are defined.

These pages include various military metrics such as:

- Aircraft Strength
- Tank Strength
- Naval Assets
- Defense Budget
- Military Personnel

These URLs will be used to download the raw HTML pages for further parsing.

In [3]:
# urls to download data
base_url = "https://www.globalfirepower.com/countries-listing.php"

other_sources = {
    'https://www.globalfirepower.com/total-population-by-country.php': 'total_population',
        'https://www.globalfirepower.com/available-military-manpower.php': 'total_military_manpower',
        'https://www.globalfirepower.com/manpower-fit-for-military-service.php': 'fit_for_service',
        'https://www.globalfirepower.com/manpower-reaching-military-age-annually.php': 'population_reaching_military_age_annually',
        'https://www.globalfirepower.com/active-military-manpower.php': 'active_personnel',
        'https://www.globalfirepower.com/active-reserve-military-manpower.php': 'reserve_personnel',
        'https://www.globalfirepower.com/manpower-paramilitary.php': 'paramilitary',
        'https://www.globalfirepower.com/aircraft-total.php': 'total_military_aircraft',
        'https://www.globalfirepower.com/aircraft-total-fighters.php': 'fighter_aircraft',
        'https://www.globalfirepower.com/aircraft-total-attack-types.php': 'attack_aircraft',
        'https://www.globalfirepower.com/aircraft-total-transports.php': 'transport_aircraft',
        'https://www.globalfirepower.com/aircraft-total-trainers.php': 'trainer_aircraft',
        'https://www.globalfirepower.com/aircraft-total-special-mission.php': 'special_mission_aircraft',
        'https://www.globalfirepower.com/aircraft-total-tanker-fleet.php': 'tanker_aircraft',
        'https://www.globalfirepower.com/aircraft-helicopters-total.php': 'total_military_helicopters',
        'https://www.globalfirepower.com/aircraft-helicopters-attack.php': 'attack_helicopters',
        'https://www.globalfirepower.com/armor-tanks-total.php': 'tanks',
        'https://www.globalfirepower.com/armor-apc-total.php': 'armored_fighting_vehicles',
        'https://www.globalfirepower.com/armor-self-propelled-guns-total.php': 'self_propelled_artillery',
        'https://www.globalfirepower.com/armor-towed-artillery-total.php': 'towed_artillery',
        'https://www.globalfirepower.com/armor-mlrs-total.php': 'rocket_projectors',
        'https://www.globalfirepower.com/navy-ships.php': 'total_naval_fleet',
        'https://www.globalfirepower.com/navy-force-by-tonnage.php': 'total_naval_fleet_tonnage_mt',
        'https://www.globalfirepower.com/navy-aircraft-carriers.php': 'aircraft_carriers',
        'https://www.globalfirepower.com/navy-helo-carriers.php': 'helicopter_carriers',
        'https://www.globalfirepower.com/navy-submarines.php': 'submarines',
        'https://www.globalfirepower.com/navy-destroyers.php': 'destroyers',
        'https://www.globalfirepower.com/navy-frigates.php': 'frigates',
        'https://www.globalfirepower.com/navy-corvettes.php': 'corvettes',
        'https://www.globalfirepower.com/navy-patrol-coastal-craft.php': 'coastal_patrol_craft',
        'https://www.globalfirepower.com/navy-mine-warfare-craft.php': 'mine_warfare_craft',
        'https://www.globalfirepower.com/defense-spending-budget.php': 'defense_budget_usd',
        'https://www.globalfirepower.com/external-debt-by-country.php': 'external_debt_usd',
        'https://www.globalfirepower.com/purchasing-power-parity.php': 'purchasing_power_parity_usd',
        'https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php': 'foreign_exchange_and_gold_reserves_usd',
        'https://www.globalfirepower.com/major-serviceable-airports-by-country.php': 'total_serviceable_airports',
        'https://www.globalfirepower.com/labor-force-by-country.php': 'labour_force',
        'https://www.globalfirepower.com/major-ports-and-terminals.php': 'major_ports_and_terminals',
        'https://www.globalfirepower.com/merchant-marine-strength-by-country.php': 'total_merchant_marine_fleet',
        'https://www.globalfirepower.com/railway-coverage.php': 'railway_coverage_km',
        'https://www.globalfirepower.com/roadway-coverage.php': 'roadway_coverage_km',
        'https://www.globalfirepower.com/oil-production-by-country.php': 'oil_production_bbl',
        'https://www.globalfirepower.com/oil-consumption-by-country.php': 'oil_consumption_bbl',
        'https://www.globalfirepower.com/proven-oil-reserves-by-country.php': 'proven_oil_reserves_bbl',
        'https://www.globalfirepower.com/natural-gas-production-by-country.php': 'natural_gas_production_cum',
        'https://www.globalfirepower.com/natural-gas-consumption-by-country.php': 'natural_gas_consumption_cum',
        'https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php': 'proven_natural_gas_reserves_cum',
        'https://www.globalfirepower.com/coal-production-by-country.php': 'coal_production_cum',
        'https://www.globalfirepower.com/coal-consumption-by-country.php': 'coal_consumption_mt',
        'https://www.globalfirepower.com/proven-coal-reserves-by-country.php': 'proven_coal_reserves_cum',
        'https://www.globalfirepower.com/square-land-area.php': 'total_land_area_sq_km',
        'https://www.globalfirepower.com/coastline-coverage.php': 'coastline_coverage_km',
        'https://www.globalfirepower.com/border-coverage.php': 'border_coverage_km',
        'https://www.globalfirepower.com/waterway-coverage.php': 'waterway_coverage_km'
}

## Step 4: Web Scraping – Downloading HTML Pages

A reusable function is created to:

1. Send an HTTP request to each URL
2. Retrieve the HTML content
3. Save the page locally for further analysis

The pages are stored inside the project directory to allow efficient parsing and debugging.

In [4]:
# Function to download the data
def download_page(url, filename):
    try:
        res = requests.get(url, timeout=15)
        if res.status_code == 200:
            with open(filename, "w", encoding="utf-8") as f:
                f.write(res.text)
        else:
            print(f"Failed: {url}")
    except Exception as e:
        print(f"Error {url} -> {e}")

In [5]:
download_page(base_url, "gfp_pages/countries_listing.html")

In [6]:
for url, name in tqdm(other_sources.items()):
    filepath = f"gfp_pages/metrics/{name}.html"
    download_page(url, filepath)
    time.sleep(2)

100%|██████████████████████████████████████████████████████████████████████████████████| 54/54 [02:51<00:00,  3.17s/it]


# Soup
## Step 5: HTML Parsing using BeautifulSoup

After downloading the pages, the HTML content is parsed using BeautifulSoup.

BeautifulSoup allows easy navigation through the HTML structure and helps extract:

- Country names
- Metric values
- Ranking information

In [7]:
from bs4 import BeautifulSoup
import pandas as pd
import os
from tqdm import tqdm
import re

In [8]:
METRIC_FOLDER = "gfp_pages/metrics"

In [9]:
files = [f for f in os.listdir(METRIC_FOLDER) if f.endswith(".html")]
files

['active_personnel.html',
 'aircraft_carriers.html',
 'armored_fighting_vehicles.html',
 'attack_aircraft.html',
 'attack_helicopters.html',
 'border_coverage_km.html',
 'coal_consumption_mt.html',
 'coal_production_cum.html',
 'coastal_patrol_craft.html',
 'coastline_coverage_km.html',
 'corvettes.html',
 'countries_listing.html',
 'defense_budget_usd.html',
 'destroyers.html',
 'external_debt_usd.html',
 'fighter_aircraft.html',
 'fit_for_service.html',
 'foreign_exchange_and_gold_reserves_usd.html',
 'frigates.html',
 'helicopter_carriers.html',
 'labour_force.html',
 'major_ports_and_terminals.html',
 'mine_warfare_craft.html',
 'natural_gas_consumption_cum.html',
 'natural_gas_production_cum.html',
 'oil_consumption_bbl.html',
 'oil_production_bbl.html',
 'paramilitary.html',
 'population_reaching_military_age_annually.html',
 'proven_coal_reserves_cum.html',
 'proven_natural_gas_reserves_cum.html',
 'proven_oil_reserves_bbl.html',
 'purchasing_power_parity_usd.html',
 'railway_co

In [10]:
with open("gfp_pages/metrics/tanks.html") as f:
    content = f.read()

In [11]:
soup = BeautifulSoup(content, "html.parser")

In [12]:
soup.find("span")

<span class="textSmall1 textLtGray">The leading annual global defense review since 2005</span>

## Step 6: Data Cleaning and Unit Conversion

Military statistics often include values with units such as:

- Million (M)
- Billion (B)
- Thousand (K)

A custom unit parser function is implemented to convert these values into standard numeric formats so that they can be used in analysis and visualization.

In [13]:
# Unit Parser
def unit(raw_text):
    if raw_text is None:
        return None

    text = raw_text.replace(",", "").strip()

    number_match = re.search(r"[\d.]+", text)
    number = number_match.group() if number_match else ""

    unit_match = re.search(r"[a-zA-Z%./]+", text)
    unit = unit_match.group() if unit_match else ""

    return f"{number} {unit}".strip()

## Step 7: Extracting Global Military Rankings

This step extracts the global military power ranking for each country.

The ranking page contains:

- Country name
- Power rank
- Power index score

This ranking data is stored in a dataframe and later merged with other military metrics.

In [14]:
def parse_power_ranking(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "lxml")

    cards = soup.find_all("div", class_="recordsetContainer")

    data = []

    for card in cards:

        # Country Name
        country_tag = card.find("div", class_="longFormName")
        if not country_tag:
            continue
        country = country_tag.get_text(strip=True)

        # Power Index
        pwr_tag = card.find("div", class_="pwrIndxContainer")
        if pwr_tag:
            text = pwr_tag.get_text(strip=True)
            match = re.search(r"[\d.]+", text)
            pwr_index = float(match.group()) if match else None
        else:
            pwr_index = None

        data.append([country, pwr_index])

    df_rank = pd.DataFrame(data, columns=["country", "p_rank"])

    return df_rank

## Step 8: Extracting Metrics

Each metric page contains multiple country entries.

The parser extracts:

- Country name
- Metric value
- Associated ranking

This information is converted into structured dataframes for each metric.

In [15]:
def parse_metric_cards(filepath, metric_name):
    with open(filepath, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "lxml")

    cards = soup.find_all("div", class_="recordsetContainer")

    data = []

    for card in cards:

        # Country Name
        country_tag = card.find("div", class_="longFormName")
        if not country_tag:
            continue
        country = country_tag.get_text(strip=True)

        # Metric Value
        value_tag = card.find("div", class_="valueContainer")
        if not value_tag:
            continue

        raw_text = value_tag.get_text(" ", strip=True)
        value = unit(raw_text)

        data.append([country, value])

    df = pd.DataFrame(data, columns=["country", metric_name])

    return df

In [16]:
# Parse ranking first
rank_file = "gfp_pages/countries_listing.html"
rank_df = parse_power_ranking(rank_file)

print("Ranking extracted:", rank_df.shape)
rank_df.head()

Ranking extracted: (145, 2)


,country,p_rank
0,United States,0.0741
1,Russia,0.0791
2,China,0.0919
3,India,0.1346
4,South Korea,0.1642


## Step 10: Exporting the Final Dataset

The final structured dataset is exported as:

Military_raw_Data.csv

This dataset will be used in the next project milestone for:

- Data cleaning
- Feature engineering
- Dashboard development
- Military comparison analytics

In [17]:
# Load all metric files
METRIC_FOLDER = "gfp_pages/metrics"
files = [f for f in os.listdir(METRIC_FOLDER) if f.endswith(".html")]

master_df = rank_df.copy()   # Start from ranking table

for file in tqdm(files):
    metric_name = file.replace(".html", "")
    filepath = os.path.join(METRIC_FOLDER, file)

    df_metric = parse_metric_cards(filepath, metric_name)

    master_df = pd.merge(master_df, df_metric, on="country", how="left")

# Sort properly by official rank
master_df = master_df.sort_values("p_rank")

print("Final Shape:", master_df.shape)
master_df.head()

100%|██████████████████████████████████████████████████████████████████████████████████| 55/55 [00:24<00:00,  2.25it/s]

Final Shape: (145, 57)


,country,p_rank,active_personnel,aircraft_carriers,armored_fighting_vehicles,attack_aircraft,attack_helicopters,border_coverage_km,coal_consumption_mt,coal_production_cum,...,total_military_helicopters,total_military_manpower,total_naval_fleet,total_naval_fleet_tonnage_mt,total_population,total_serviceable_airports,towed_artillery,trainer_aircraft,transport_aircraft,waterway_coverage_km
0,United States,0.0741,1333030,11,409660,926,1024,12002 km,495156000 mt,534234000 Cu.M,...,5913,150463900,465,8265799,341963408,16116,1878,2610,917,41009 km
1,Russia,0.0791,1320000,1,126512,698,556,22407 km,290763000 mt,531130000 Cu.M,...,1643,69002197,747,1426539,140820810,905,5920,530,458,102000 km
2,China,0.0919,2035000,3,152040,371,281,22457 km,5191000000 mt,4805000000 Cu.M,...,1007,764123366,841,3192411,1415043270,552,1400,401,287,27700 km
3,India,0.1346,1431000,2,163554,124,79,13888 km,1262000000 mt,1020000000 Cu.M,...,594,662290299,343,631989,1409128296,315,5640,334,277,14500 km
4,South Korea,0.1642,450000,0,117460,98,113,237 km,136817000 mt,16081000 Cu.M,...,827,26040900,215,427946,52081799,92,5800,335,40,1600 km


In [18]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 145 entries, 0 to 144
Data columns (total 57 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   country                                    145 non-null    object 
 1   p_rank                                     145 non-null    float64
 2   active_personnel                           145 non-null    object 
 3   aircraft_carriers                          145 non-null    object 
 4   armored_fighting_vehicles                  145 non-null    object 
 5   attack_aircraft                            145 non-null    object 
 6   attack_helicopters                         145 non-null    object 
 7   border_coverage_km                         145 non-null    object 
 8   coal_consumption_mt                        145 non-null    object 
 9   coal_production_cum                        145 non-null    object 
 10  coastal_patrol_craft           

In [19]:
master_df.to_csv("Military_raw_Data.csv", index=False)
print("Military_raw_Data.csv saved successfully")

Military_raw_Data.csv saved successfully
